## Task Overview

This task applies **Rice coding** to the residual signal produced by a **first-order predictor** for two WAV audio files.

The process begins by loading each source file and converting the audio samples into a one-dimensional integer sequence. A first-order predictor is then used, where each sample is predicted from the previous one. Instead of encoding the original waveform directly, the difference between consecutive samples is stored as a residual. For many audio signals, these residuals are smaller in magnitude than the original samples, which makes them more suitable for entropy coding.

Because Rice coding operates on non-negative integers, the signed residual values are first mapped into unsigned form using a reversible transformation. The residuals are then encoded using two different Rice parameters, **k = 2** and **k = 4**, so that the effect of the coding choice can be compared.

After encoding, the bitstream is decoded back into residuals, and the original signal is reconstructed from those residuals. A lossless check is then performed by comparing the reconstructed samples against the original flattened audio data.

Finally, the encoded bitstream and the decoded WAV file are saved for each setting, and the notebook reports the encoded size, compression percentage, and whether the process remained fully lossless.

In [1]:
import os
import numpy as np
from scipy.io import wavfile

# Define the input directory that contains the original WAV files.
INPUT_DIR = os.path.join("..", "Exercise2_Files")

# Define the output directory where encoded and decoded results will be saved.
OUTPUT_DIR = os.path.join("..", "Exercise2")

# Create the output directory if it does not already exist.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# List the source audio files to be processed.
FILES = ["Sound1.wav", "Sound2.wav"]

# Define the Rice parameters to test.
K_VALUES = [2, 4]

### Importing libraries and setting up file paths

This cell imports the libraries needed for the audio compression task.

- `os` is used for file and folder handling.
- `numpy` is used for numerical operations on the audio samples.
- `wavfile` from `scipy.io` is used to read and write WAV audio files.

The input and output directories are then defined, and the output folder is created if it does not already exist. The code also lists the audio files to process and the Rice parameter values that will be tested.

In [2]:
# ---------------------------------------------------------
# Signed <-> unsigned mapping for residuals
# ---------------------------------------------------------
def signed_to_unsigned(n: int) -> int:
    # Convert a signed integer into a non-negative value for Rice coding.
    n = int(n)
    return 2 * n if n >= 0 else -2 * n - 1


def unsigned_to_signed(u: int) -> int:
    # Recover the original signed integer from the mapped unsigned value.
    u = int(u)
    return u // 2 if (u % 2 == 0) else -(u // 2) - 1

### Mapping signed residuals into unsigned values

This cell defines the conversion between signed and unsigned integers.

Residual values can be positive or negative, but Rice coding works with non-negative integers. The first function maps signed values into unsigned form in a reversible way, so that small positive and negative values remain close to zero. The second function restores the original signed value during decoding.

This step is necessary because the residual signal produced by prediction is naturally signed.

In [3]:
# ---------------------------------------------------------
# Predictor
# ---------------------------------------------------------
def compute_residuals(samples: np.ndarray) -> np.ndarray:
    # Convert the input signal to a flat 32-bit integer array.
    samples = np.asarray(samples, dtype=np.int32).reshape(-1)

    # Reserve space for the residual sequence.
    residuals = np.empty_like(samples)

    # Store the first sample directly because there is no previous sample to predict from.
    residuals[0] = samples[0]

    # Use first-order prediction: current sample minus previous sample.
    residuals[1:] = samples[1:] - samples[:-1]
    return residuals


def reconstruct_from_residuals(residuals: np.ndarray) -> np.ndarray:
    # Convert the residual sequence to a flat 32-bit integer array.
    residuals = np.asarray(residuals, dtype=np.int32).reshape(-1)

    # Reserve space for the reconstructed sample sequence.
    samples = np.empty_like(residuals)

    # Restore the first sample directly.
    samples[0] = residuals[0]

    # Rebuild the waveform cumulatively from the residuals.
    for i in range(1, len(residuals)):
        samples[i] = samples[i - 1] + residuals[i]

    return samples

### Computing and reconstructing prediction residuals

This cell defines the predictor stage.

`compute_residuals()` applies a **first-order predictor**, where each sample is predicted from the previous one. The first sample is stored directly, and every later residual is calculated as the difference between the current sample and the previous sample.

`reconstruct_from_residuals()` performs the reverse operation. Starting from the first stored value, it rebuilds the original waveform by cumulatively adding each residual.

This approach is useful because neighbouring audio samples are often similar, so the residual values are usually smaller than the original waveform values.

In [4]:
# ---------------------------------------------------------
# Rice encode / decode
# ---------------------------------------------------------
def rice_encode(values: np.ndarray, k: int) -> str:
    # Reject invalid Rice parameters.
    if k < 0:
        raise ValueError("k must be >= 0")

    # Compute m = 2^k, which controls quotient and remainder splitting.
    m = 1 << k
    bit_parts = []

    # Encode each residual value one at a time.
    for val in np.asarray(values).reshape(-1):
        # Convert the signed residual to an unsigned value.
        u = signed_to_unsigned(int(val))

        # Split the value into quotient and remainder.
        q = u // m
        r = u % m

        # Build the Rice codeword.
        if k == 0:
            code = ("1" * q) + "0"
        else:
            code = ("1" * q) + "0" + format(r, f"0{k}b")

        bit_parts.append(code)

    # Join all codewords into one continuous bitstream.
    return "".join(bit_parts)


def rice_decode(bitstream: str, k: int, count: int) -> np.ndarray:
    # Reject invalid Rice parameters.
    if k < 0:
        raise ValueError("k must be >= 0")

    # Compute m = 2^k for decoding.
    m = 1 << k
    out = []
    i = 0

    # Decode exactly the requested number of values.
    for _ in range(count):
        q = 0

        # Read the unary-coded quotient.
        while i < len(bitstream) and bitstream[i] == "1":
            q += 1
            i += 1

        if i >= len(bitstream):
            raise ValueError("Bitstream ended while decoding unary quotient")

        # Skip the terminating zero after the unary quotient.
        i += 1

        # Read the fixed-length binary remainder.
        if k == 0:
            r = 0
        else:
            if i + k > len(bitstream):
                raise ValueError("Bitstream ended while decoding remainder")
            r = int(bitstream[i:i + k], 2)
            i += k

        # Recover the unsigned value and convert it back to signed form.
        u = q * m + r
        out.append(unsigned_to_signed(u))

    return np.array(out, dtype=np.int32)

### Rice encoding and decoding

This cell implements the Rice coder itself.

`rice_encode()` converts each residual into its Rice codeword using the selected value of `k`. The mapped unsigned value is split into a quotient and remainder. The quotient is written in unary form, and the remainder is written in binary using `k` bits. All codewords are then joined into one bitstream string.

`rice_decode()` performs the reverse process. It reads the unary quotient, then the fixed-length remainder, reconstructs the original unsigned value, and converts it back into a signed residual.

Together, these two functions form the main compression and decompression stage of the task.

In [5]:
# ---------------------------------------------------------
# Save / load encoded bitstream
# ---------------------------------------------------------
def save_bitstream(path: str, bitstream: str):
    # Save the encoded bitstream as plain text.
    with open(path, "w", encoding="utf-8") as f:
        f.write(bitstream)

### Saving the encoded bitstream

This cell defines a small helper function for storing the encoded Rice bitstream.

The bitstream is written as text to the output file so that the encoded result can be inspected and saved separately from the decoded audio. This keeps the encoded output available for size comparison and verification.

In [6]:
# ---------------------------------------------------------
# Main processing
# ---------------------------------------------------------
for file_name in FILES:
    # Build the full path to the current input WAV file.
    in_path = os.path.join(INPUT_DIR, file_name)

    # Skip missing files while continuing with any remaining inputs.
    if not os.path.exists(in_path):
        print(f"Missing file: {in_path}")
        continue

    # Read the WAV file and obtain the sampling rate and audio samples.
    fs, audio = wavfile.read(in_path)

    # Accept only 16-bit PCM audio for this implementation.
    if audio.dtype != np.int16:
        raise ValueError(f"{file_name}: expected int16 PCM, got {audio.dtype}")

    # Keep the original shape so the reconstructed audio can be restored correctly.
    original_shape = audio.shape

    # Flatten the audio into a one-dimensional int32 array for processing.
    flat_audio = audio.reshape(-1).astype(np.int32)

    # Compute the original size in bits.
    original_bits = flat_audio.size * 16

    # Generate first-order prediction residuals.
    residuals = compute_residuals(flat_audio)

    print(f"\n--- {file_name} ---")

    for k in K_VALUES:
        # Encode the residuals using Rice coding with the current k value.
        encoded_bits = rice_encode(residuals, k)
        encoded_size_bits = len(encoded_bits)

        # Compute the compression percentage relative to the original PCM size.
        compression_percent = (1 - encoded_size_bits / original_bits) * 100.0

        # Decode the bitstream and reconstruct the original signal.
        decoded_residuals = rice_decode(encoded_bits, k, len(residuals))
        reconstructed = reconstruct_from_residuals(decoded_residuals)

        # Check whether reconstruction is exactly identical to the original signal.
        identical = np.array_equal(flat_audio, reconstructed)

        # Restore the decoded signal to its original shape and 16-bit format.
        reconstructed_audio = reconstructed.reshape(original_shape).astype(np.int16)

        # Build filenames for the encoded and decoded outputs.
        base = os.path.splitext(file_name)[0]
        encoded_path = os.path.join(OUTPUT_DIR, f"{base}_k{k}.ex2")
        decoded_wav_path = os.path.join(OUTPUT_DIR, f"{base}_k{k}_decoded.wav")

        # Save the encoded bitstream and the reconstructed WAV file.
        save_bitstream(encoded_path, encoded_bits)
        wavfile.write(decoded_wav_path, fs, reconstructed_audio)

        # Print the compression result and output locations.
        print(
            f"k={k}: "
            f"Encoded={encoded_size_bits} bits | "
            f"Compression={compression_percent:.2f}% | "
            f"Lossless={identical}"
        )
        print(f"  Saved encoded: {encoded_path}")
        print(f"  Saved decoded: {decoded_wav_path}")


--- Sound1.wav ---
k=2: Encoded=19439238 bits | Compression=-142.49% | Lossless=True
  Saved encoded: ../Exercise2/Sound1_k2.ex2
  Saved decoded: ../Exercise2/Sound1_k2_decoded.wav
k=4: Encoded=6859604 bits | Compression=14.43% | Lossless=True
  Saved encoded: ../Exercise2/Sound1_k4.ex2
  Saved decoded: ../Exercise2/Sound1_k4_decoded.wav

--- Sound2.wav ---
k=2: Encoded=1809572852 bits | Compression=-22340.14% | Lossless=True
  Saved encoded: ../Exercise2/Sound2_k2.ex2
  Saved decoded: ../Exercise2/Sound2_k2_decoded.wav
k=4: Encoded=454346298 bits | Compression=-5534.25% | Lossless=True
  Saved encoded: ../Exercise2/Sound2_k4.ex2
  Saved decoded: ../Exercise2/Sound2_k4_decoded.wav


### Running the full encoding and decoding process

This final cell applies the whole workflow to each audio file.

For every WAV file, the code reads the signal, checks that it is 16-bit PCM audio, flattens the samples, and computes the first-order residual sequence. The original PCM size in bits is also calculated for later comparison.

The residuals are then encoded using both `k = 2` and `k = 4`. For each case, the code measures the encoded size, calculates the compression percentage, decodes the bitstream again, and reconstructs the waveform from the decoded residuals.

A lossless check is performed by comparing the reconstructed signal with the original flattened signal. The reconstructed audio is then reshaped back to its original format and saved as a decoded WAV file, while the encoded bitstream is saved as an `.ex2` file.

The printed output summarises the encoded size, compression percentage, and whether the reconstruction remained exactly lossless for each file and each Rice parameter.